In [1]:
# on va donc enlever les lignes : 
import os
import ast
import numpy as np
import pandas as pd

def build_daily(p):
    base = f"../data/Test_technique/{p}/"
    fb, pm, gd = base + "fitbit/", base + "pmsys/", base + "googledocs/"

    # tableau vide si le fichier existe pas (ex : p03 sans heart_rate)
    def vide(cols):
        return pd.DataFrame(columns=cols)

    # pas/calories /distance -> somme par jour
    def per_minute(file, name):
        if not os.path.exists(fb + file):
            return pd.Series(name=name, dtype=float)
        df = pd.read_json(fb + file)
        df["date"] = pd.to_datetime(df["dateTime"]).dt.date
        return df.groupby("date")["value"].sum().rename(name)

    steps = per_minute("steps.json", "steps")
    calories = per_minute("calories.json", "calories")
    distance = per_minute("distance.json", "distance_km") / 100000  # cm -> km

    # fc : p05 au lieu du min 
    if os.path.exists(fb + "heart_rate.json"):
        hr = pd.read_json(fb + "heart_rate.json")
        hr["bpm"] = hr["value"].apply(lambda d: d["bpm"])
        hr["date"] = pd.to_datetime(hr["dateTime"]).dt.date
        hr_day = hr.groupby("date")["bpm"].agg(
            hr_mean="mean", hr_p05=lambda x: x.quantile(0.05), hr_max="max"
        )
    else:
        hr_day = vide(["hr_mean", "hr_p05", "hr_max"])

    # fc repos, 0 = pas calculée
    if os.path.exists(fb + "resting_heart_rate.json"):
        rest = pd.read_json(fb + "resting_heart_rate.json")
        rest["date"] = pd.to_datetime(rest["dateTime"]).dt.date
        rest["rhr"] = rest["value"].apply(lambda d: d["value"])
        rhr = rest.groupby("date")["rhr"].first().replace(0, np.nan)
    else:
        rhr = pd.Series(name="rhr", dtype=float)

    # sommeil, nuit principale seulement + doublons virés
    cols_sleep = ["sleep_minutes", "sleep_efficiency", "sleep_deep", "sleep_light", "sleep_rem", "sleep_wake"]
    if os.path.exists(fb + "sleep.json"):
        sleep = pd.read_json(fb + "sleep.json")
        main = sleep[sleep["mainSleep"] == True].drop_duplicates(
            subset=["dateOfSleep", "startTime", "endTime"]).copy()
        main["date"] = pd.to_datetime(main["dateOfSleep"]).dt.date
        levels = pd.json_normalize(main["levels"].apply(lambda d: d["summary"]))
        levels.index = main.index
        phases = levels.reindex(columns=["deep.minutes", "light.minutes", "rem.minutes", "wake.minutes"])
        phases.columns = ["sleep_deep", "sleep_light", "sleep_rem", "sleep_wake"]
        sleep_day = pd.concat([main[["date", "minutesAsleep", "efficiency"]], phases], axis=1)
        sleep_day = sleep_day.rename(columns={"minutesAsleep": "sleep_minutes", "efficiency": "sleep_efficiency"})
        sleep_day = sleep_day.groupby("date").first()
    else:
        sleep_day = vide(cols_sleep)

    # sleep score
    cols_score = ["overall_score", "composition_score", "revitalization_score", "duration_score", "restlessness"]
    if os.path.exists(fb + "sleep_score.csv"):
        score = pd.read_csv(fb + "sleep_score.csv")
        score["date"] = pd.to_datetime(score["timestamp"]).dt.date
        score_day = score.drop_duplicates("date").set_index("date")[cols_score]
    else:
        score_day = vide(cols_score)

    # zones cardio
    if os.path.exists(fb + "time_in_heart_rate_zones.json"):
        zones_raw = pd.read_json(fb + "time_in_heart_rate_zones.json")
        zones = pd.json_normalize(zones_raw["value"])
        zones.index = pd.to_datetime(zones_raw["dateTime"]).dt.date
        zones.columns = [c.replace("valuesInZones.", "") for c in zones.columns]
        zones = zones.rename(columns={
            "BELOW_DEFAULT_ZONE_1": "zone_below", "IN_DEFAULT_ZONE_1": "zone_fatburn",
            "IN_DEFAULT_ZONE_2": "zone_cardio", "IN_DEFAULT_ZONE_3": "zone_peak"})
        zones = zones.groupby(level=0).first()
    else:
        zones = vide(["zone_below", "zone_fatburn", "zone_cardio", "zone_peak"])

    # wellness
    cols_well = ["fatigue", "mood", "readiness", "sleep_duration_h", "sleep_quality", "soreness", "stress"]
    if os.path.exists(pm + "wellness.csv"):
        well = pd.read_csv(pm + "wellness.csv")
        well["date"] = pd.to_datetime(well["effective_time_frame"]).dt.date
        well_day = well.drop_duplicates("date", keep="last").set_index("date")[cols_well]
    else:
        well_day = vide(cols_well)

    # srpe, charge = rpe * durée
    if os.path.exists(pm + "srpe.csv"):
        srpe = pd.read_csv(pm + "srpe.csv")
        srpe["date"] = pd.to_datetime(srpe["end_date_time"]).dt.date
        srpe["load"] = srpe["perceived_exertion"] * srpe["duration_min"]
        srpe_day = srpe.groupby("date").agg(
            train_load=("load", "sum"), train_minutes=("duration_min", "sum"), n_sessions=("load", "size"))
    else:
        srpe_day = vide(["train_load", "train_minutes", "n_sessions"])

    # blessures (texte -> dict)
    if os.path.exists(pm + "injury.csv"):
        inj = pd.read_csv(pm + "injury.csv")
        inj["date"] = pd.to_datetime(inj["effective_time_frame"]).dt.date
        inj["inj_dict"] = inj["injuries"].apply(ast.literal_eval)
        inj["injured"] = (inj["inj_dict"].apply(len) > 0).astype(int)
        inj["major"] = inj["inj_dict"].apply(lambda d: int("major" in d.values()))
        inj_day = inj.drop_duplicates("date", keep="last").set_index("date")[["injured", "major"]]
    else:
        inj_day = vide(["injured", "major"])

    # reporting, attention dates en jj/mm
    cols_rep = ["n_meals", "weight", "glasses_of_fluid", "alcohol"]
    if os.path.exists(gd + "reporting.csv"):
        rep = pd.read_csv(gd + "reporting.csv")
        rep["date"] = pd.to_datetime(rep["date"], dayfirst=True).dt.date
        rep["n_meals"] = rep["meals"].str.split(",").str.len()
        rep["alcohol"] = (rep["alcohol_consumed"] == "Yes").astype(int)
        rep_day = rep.drop_duplicates("date", keep="last").set_index("date")[cols_rep]
    else:
        rep_day = vide(cols_rep)

    daily = pd.concat([steps, calories, distance, hr_day, rhr, sleep_day, score_day,
                       zones, well_day, srpe_day, inj_day, rep_day], axis=1)
    # calendrier complet, les jours sans montre restent en lignes vides
    periode = pd.date_range(min(steps.index), max(steps.index)).date
    daily = daily.reindex(periode)

    # pas de séance = repos
    cols_train = ["train_load", "train_minutes", "n_sessions"]
    daily[cols_train] = daily[cols_train].fillna(0)

    daily.index.name = "date"
    daily = daily.reset_index()
    daily.insert(0, "participant", p)
    return daily


In [2]:
dataset = pd.concat([build_daily(p) for p in ["p01", "p03", "p05"]], ignore_index=True)
print(dataset.shape)
display(dataset.groupby("participant").size())
os.makedirs("../data/processed", exist_ok=True)
dataset.to_csv("../data/processed/daily_raw.csv", index=False)


(456, 40)


participant
p01    152
p03    152
p05    152
dtype: int64

In [3]:
#Verif
dataset["date"] = pd.to_datetime(dataset["date"])
dataset.groupby("participant")["date"].agg(["min", "max", "count"])

,min,max,count
participant,,,
p01,2019-11-01,2020-03-31,152
p03,2019-11-01,2020-03-31,152
p05,2019-11-01,2020-03-31,152


In [4]:
for p, d in dataset.groupby("participant"):
    vides = d[d["steps"].isna()]["date"]
    print(p, len(vides), list(vides.dt.date))

p01 0 []
p03 30 [datetime.date(2019, 11, 4), datetime.date(2019, 11, 10), datetime.date(2019, 11, 17), datetime.date(2019, 11, 18), datetime.date(2019, 11, 19), datetime.date(2019, 11, 20), datetime.date(2019, 11, 21), datetime.date(2019, 11, 22), datetime.date(2019, 11, 23), datetime.date(2019, 11, 24), datetime.date(2019, 11, 25), datetime.date(2019, 11, 26), datetime.date(2019, 11, 27), datetime.date(2019, 11, 28), datetime.date(2019, 11, 29), datetime.date(2019, 11, 30), datetime.date(2019, 12, 1), datetime.date(2019, 12, 2), datetime.date(2019, 12, 3), datetime.date(2019, 12, 4), datetime.date(2019, 12, 5), datetime.date(2019, 12, 6), datetime.date(2019, 12, 7), datetime.date(2019, 12, 8), datetime.date(2019, 12, 9), datetime.date(2020, 2, 29), datetime.date(2020, 3, 1), datetime.date(2020, 3, 2), datetime.date(2020, 3, 3), datetime.date(2020, 3, 28)]
p05 8 [datetime.date(2019, 11, 11), datetime.date(2020, 2, 8), datetime.date(2020, 2, 22), datetime.date(2020, 2, 23), datetime.dat

In [5]:
# Jours sans montre : 0 pour p1, 30 pour p3 dont un long bloc de 23 jours , 8 pour p5 dont un bloc de 6 jours
# Ce qui peut nous aider par la suite, car on pourra interpolé les valeurs ne faisant pas partis de ces longs blocs


In [6]:
#Valeurs manquantes :
dataset.groupby("participant").apply(lambda d: d.isna().mean().round(2)).T

participant,p01,p03,p05
date,0.00,0.00,0.00
steps,0.00,0.20,0.05
calories,0.00,0.00,0.00
distance_km,0.00,0.20,0.05
hr_mean,0.00,1.00,0.05
hr_p05,0.00,1.00,0.05
hr_max,0.00,1.00,0.05
rhr,0.00,0.34,0.43
sleep_minutes,0.01,0.49,0.22
sleep_efficiency,0.01,0.49,0.22


In [7]:
# On voit que les calories reste a 0 ( donc pas de valeurs manquantes ), alors que les steps manque pour p3 dans 20% du temps et pour p5 dans 5% , donc la montre "produit" des calories
# même quand elle n'enregistre aucun pas ce qui est étrange , peut être un fonctionnement de fitbit sur une estimation , à voir .

In [8]:
dataset[dataset["steps"].isna()].groupby("participant")["calories"].describe()

,count,mean,std,min,25%,50%,75%,max
participant,,,,,,,,
p03,30.0,1940.16,6.476781,1929.6,1933.2,1944.0,1944.0,1944.0
p05,8.0,1931.40,5.091169,1929.6,1929.6,1929.6,1929.6,1944.0


In [9]:
# correction : calories estimées les jours sans montre -> NaN
dataset.loc[dataset["steps"].isna(), "calories"] = np.nan
dataset.groupby("participant")["calories"].apply(lambda x: x.isna().mean().round(2))

participant
p01    0.00
p03    0.20
p05    0.05
Name: calories, dtype: float64

In [10]:
os.makedirs("../data/processed", exist_ok=True)
dataset.to_csv("../data/processed/daily_raw.csv", index=False)